In [1]:
import numpy as np
import pandas as pd
import os
import pickle as pkl

In [2]:
file_list = os.listdir(os.getcwd())

In [3]:
save_df = pd.read_csv('sum_dat.csv')

In [4]:
def classify_congruence(row):
    if row['response'] == 'down' and row['stimuli_type'] == 2:
        return .5
    elif row['response'] == 'up' and row['stimuli_type'] == 1:
        return .5
    else:
        return -.5

In [5]:
save_df['cong'] = save_df.apply(classify_congruence,axis=1)

In [26]:
## select session 2-4
select_df = save_df[save_df['session'] != 0].dropna()
select_df = select_df[select_df['subjectively_correct']!='Late']
select_df['unit'] = select_df['sub_num'].astype(str) + '_ses-' + select_df['session'].astype(str)
codes, uniques = pd.factorize(select_df['unit'], sort=False)

In [27]:
save_fit = {
    'ns':len(uniques),
    'nt':len(codes),
    'sub_list':codes.astype('int')+1,
    'choice':select_df['response'].map({'up':0,'down':1}).astype('int'),
    'vol':((select_df['happ_vol_stab']+select_df['ang_vol_stab'])>.1)*1- .5,
    'cue':select_df['valence'].map({'Angry':-.5,'Happy':.5}),
    'cong':select_df['cong'],
    'outcome':select_df['subjectively_correct'].map({'False':0,'True':1}).astype('int')
    
}

In [28]:
with open('stan_fit.pkl','wb') as f:
    pkl.dump(save_fit,f)